In [ ]:
(* load the setup of the notebooks *)
#use "../setup.ml";;

The module `RSet`, which is loaded by `setup.ml`, provides an implementation of mathematical sets that can be nested.  The type of sets with elements of type `'a` is `'a RSet.t`.

# The Usual Suspects
---
There has been a burglary at a jewelry store.  The [usual suspects](https://en.wikipedia.org/wiki/The_Usual_Suspects) have been arrested.  These are
 * Aaron,
 * Bernard, and
 * Caine.

Furthermore, the following facts have been established:
 1. It is known that at least one of these suspects is indeed guilty.
 2. If Aaron is guilty, he has exactly one accomplice.
 3. If Bernard is innocent, then Caine is inncocent, too.
 4. If exactly two of the suspects are guilty, then Caine is one of them.
 5. If Caine is innocent, then Aaron is guilty.

It is our task to identify those suspects that are guilty.

Our first task is to define the set of propositional variables:
$$ \mathcal{P} := \{ \texttt{a}, \texttt{b}, \texttt{c} \} $$
The interpretation is that
 * $\texttt{a}$ is true iff Aaron is guilty,
 * $\texttt{b}$ is true iff Bernard is guilty, and
 * $\texttt{c}$ is true iff Caine is guilty.

In <samp>OCaml</samp>, the names of variables have to start with a lower case letter.  Therefore, we store the set $\mathcal{P}$ in the variable `vars`.

In [ ]:
let vars = RSet.of_list ["a"; "b"; "c"]

Our next task is to translate the facts given above into formulas from propositional logic.

The statement "*It is known that at least one of these suspects is indeed guilty.*" is translated as follows:
$$
\texttt{a} \vee \texttt{b} \vee \texttt{c}.
$$

In [ ]:
let f1 = "a ∨ b ∨ c"

The statement "*If Aaron is guilty, he has exactly one accomplice.*" is harder to translate into propositional logic. The idea is to split this statement into two statements:
* If Aaron is guilty, he has at least one accomplice.
* If Aaron is guilty, he has at most  one accomplice.

These statements can now be translated into the following formulas:

In [ ]:
let f2 = "a → b ∨ c"

In [ ]:
let f3 = "a → ¬(b ∧ c)"

The statement "*If Bernard is innocent, then Caine is inncocent, too.*" is a straightforward implication:

In [ ]:
let f4 = "¬b → ¬c"

The statement "*If exactly two of the suspects are guilty, then Caine is one of them.*" is best translated into propositional logic by asking how this statement could be made false.

This statement is false if and only if
  + two suspects are guilty,
  + but Caine is innocent.

However this is only possible if and only if
  + *Caine is innocent*,
  + *Aaron and Bernard are guilty*.

Hence we can translate this statement as follows:

In [ ]:
let f5 = "¬(¬c ∧ a ∧ b)"

The statement "*If Caine is innocent, then Aaron is guilty.*" is an implication:

In [ ]:
let f6 = "¬c → a"

We define the list `fs` of all formulas:

In [ ]:
let fs = [f1; f2; f3; f4; f5; f6]

We need to transform the strings <tt>f1</tt> to <tt>f6</tt> into values of type `formula`.  To this end we import a parser for propositional formulas.

In [ ]:
import_notebook "Propositional-Logic-Parser.ipynb"

Next, we transform all formulas into values of type `formula`:

In [ ]:
let gs = List.map parse fs

We are looking for a variable assignment $\mathcal{I}$ that satisfies all formulas in the list <tt>fs</tt>.  As variable assignments are represented as subsets of the set $\mathcal{P}$ of propositional variables, we can just iterate over all subsets of $\mathcal{P}$.

The function `evaluate` takes two arguments:
- `f` is a propositional formula,
- `i` is a propositional variable assignment, i.e. the set of those variables that are true.

It evaluates `f` using the assignment `i`.  We have discussed the details of this function previously.

In [ ]:
let rec evaluate (f : formula) (i : string RSet.t) : bool =
  match f with
  | Var p      -> RSet.mem p i
  | Verum      -> true
  | Falsum     -> false
  | Not g      -> not (evaluate g i)
  | And (g, h) -> evaluate g i && evaluate h i
  | Or  (g, h) -> evaluate g i || evaluate h i
  | Imp (g, h) -> evaluate g i <= evaluate h i
  | Iff (g, h) -> evaluate g i =  evaluate h i

The function `all_true` takes two arguments:
- `gs` is a list of propositional formulas,
- `i` is a propositional variable assignment.

It returns `true` if and only if all formulas from `gs` are `true` given the variable assignment `i`.  `List.for_all p l` checks whether every element of the list `l` satisfies the predicate `p`.

In [ ]:
let all_true (gs : formula list) (i : string RSet.t) : bool =
  List.for_all (fun f -> evaluate f i) gs

The function `RSet.powerset` returns the *power set* of a given set, i.e. the set of all subsets of the given set.

In [ ]:
RSet.powerset vars

Next, we compute the set of all variable assignments that render all formulas true.  As `all_true` is a curried function, the partial application `all_true gs` is a predicate that can be passed to `RSet.filter`.

In [ ]:
let valid_assignments = RSet.filter (all_true gs) (RSet.powerset vars)

It turns our that there is just one propositional variable assignment that satisfies all formulas from the list <tt>fs</tt>.  Therefore, the problem has a unique solution: Bernard and Caine are guilty, while Aaron is innocent.